## Phase 7 · Evaluation — Environment Setup

This is a NEW notebook = a fresh machine. Nothing from the training notebook
carries over, so we reinstall Unsloth (which brings transformers, PEFT, TRL,
bitsandbytes, accelerate) and will re-authenticate to Hugging Face in the next
cell. We then load the base model + our trained adapter *from the Hub* — we do
NOT retrain here. `sqlite3` (for running SQL) ships with Python.

In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 767.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 M

### Re-authenticate to Hugging Face

Fresh notebook = fresh session, so we log in again. The `HF_TOKEN` secret from
before still exists (secrets are per-account), but this notebook needs its own
access toggle enabled (🔑 sidebar → HF_TOKEN → Notebook access ON). We need to
be logged in to pull our adapter from the Hub in the next step.

In [2]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get('HF_TOKEN'))
print("Logged in as:", whoami()["name"])

Logged in as: nikhil1772000


### Load base Qwen + attach the fine-tuned adapter (from the Hub)

We load our adapter repo directly — Unsloth reads its config, pulls the matching
base Qwen2.5-Coder-3B in 4-bit, and attaches the adapter. `for_inference()`
enables the fast generation path. For evaluation we'll toggle the adapter
OFF (base model) vs ON (fine-tuned) to compare the two on the same questions.

In [3]:
from unsloth import FastLanguageModel

REPO_ID = "nikhil1772000/text2sql-qlora-qwen2.5-coder-3b"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = REPO_ID,     # our adapter; base is pulled automatically
    max_seq_length = 2048,
    dtype         = None,
    load_in_4bit  = True,
)

FastLanguageModel.for_inference(model)   # enable fast inference
print("Fine-tuned model loaded and ready for inference.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Fine-tuned model loaded and ready for inference.


### Download the Spider dev databases (the real .sqlite files)

Execution accuracy = run both the predicted and gold SQL against the actual
database and compare result rows. That needs the populated `.sqlite` files, not
just schema text. We `snapshot_download` the Spider dataset repo, then discover
where the `.sqlite` files are (handling a possible zip). We only evaluate on the
dev databases — the 20 the model never saw in training.

In [4]:
from huggingface_hub import snapshot_download
import os, glob, zipfile

# Pull the dataset repo (databases live here alongside the parquet)
local_dir = snapshot_download(
    repo_id   = "xlangai/spider",
    repo_type = "dataset",
    local_dir = "spider_data",
)
print("Downloaded repo to:", local_dir)

# Find .sqlite files; if none yet, unzip any archives and look again
def find_sqlite(root):
    return glob.glob(os.path.join(root, "**", "*.sqlite"), recursive=True)

sqlite_files = find_sqlite(local_dir)
if not sqlite_files:
    for z in glob.glob(os.path.join(local_dir, "**", "*.zip"), recursive=True):
        print("Unzipping:", z)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(local_dir)
    sqlite_files = find_sqlite(local_dir)

print("Total .sqlite files found:", len(sqlite_files))
print("A few examples:")
for f in sqlite_files[:5]:
    print("  ", f)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Downloaded repo to: /content/spider_data
Total .sqlite files found: 0
A few examples:


### Download the official Spider databases (populated .sqlite files)

The parquet/JSON mirrors strip the data rows, but execution accuracy needs
populated databases. We fetch the official Yale Spider distribution (a Google
Drive zip that includes `database/` with all the .sqlite files + `dev.json`)
using `gdown`, unzip it, then verify real .sqlite files appeared. If the Drive
ID is stale the cell will say so clearly and we'll switch sources.

In [5]:
!pip install -q gdown
import gdown, zipfile, os, glob

# Official Spider distribution (repackaged "spider_data") on Google Drive
SPIDER_GDRIVE_ID = "1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J"

try:
    gdown.download(id=SPIDER_GDRIVE_ID, output="spider_data.zip", quiet=False)
    print("Download call finished. File size (MB):",
          round(os.path.getsize("spider_data.zip") / 1e6, 1))
except Exception as e:
    print("gdown failed:", repr(e))

# Unzip if we got a real archive
if os.path.exists("spider_data.zip") and os.path.getsize("spider_data.zip") > 1_000_000:
    with zipfile.ZipFile("spider_data.zip") as zf:
        zf.extractall("spider_db")
    sqlite_files = glob.glob("spider_db/**/*.sqlite", recursive=True)
    print("\n.sqlite files found:", len(sqlite_files))
    for f in sqlite_files[:5]:
        print("  ", f)
else:
    print("\nNo valid zip downloaded — the Drive ID is likely stale. "
          "Paste this output and we'll switch to an alternate source.")

Downloading...
From (original): https://drive.google.com/uc?id=1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J
From (redirected): https://drive.google.com/uc?id=1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J&confirm=t&uuid=173fab5f-5df2-459f-af3b-bf0a37ba72f6
To: /content/spider_data.zip
100%|██████████| 206M/206M [00:01<00:00, 191MB/s]


Download call finished. File size (MB): 205.8

.sqlite files found: 372
   spider_db/spider_data/database/gymnast/gymnast.sqlite
   spider_db/spider_data/database/student_transcripts_tracking/student_transcripts_tracking.sqlite
   spider_db/spider_data/database/customer_deliveries/customer_deliveries.sqlite
   spider_db/spider_data/database/insurance_fnol/insurance_fnol.sqlite
   spider_db/spider_data/database/product_catalog/product_catalog.sqlite


### Map db_id → .sqlite path, and build the SQL execution helper

`run_query(sql, db_id)` connects to the right database, executes the SQL, and
returns the result rows as a **set** of tuples (unordered — the standard for
execution accuracy). On any SQL error it returns `None` (a sentinel) instead of
crashing, so invalid model output can't halt the whole eval loop. We test it on
one known example (gold query on its database) before using it at scale.

In [6]:
import sqlite3, glob, os

# Build db_id -> .sqlite path map (prefer the main `database/` dir)
db_paths = {}
for f in glob.glob("spider_db/**/*.sqlite", recursive=True):
    db_id = os.path.splitext(os.path.basename(f))[0]
    # Prefer a path under /database/ over /test_database/ if both exist
    if db_id not in db_paths or "/database/" in f:
        db_paths[db_id] = f

print("Databases mapped:", len(db_paths))

def run_query(sql, db_id):
    """Run SQL on db_id's database. Return set of result rows, or None on error."""
    path = db_paths.get(db_id)
    if path is None:
        return None
    try:
        conn = sqlite3.connect(path)
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        conn.close()
        return set(rows)
    except Exception:
        return None

# Test on one known dev example: gold query should return real rows
test_db = "concert_singer"          # a standard Spider dev database
test_sql = "SELECT count(*) FROM singer"
print("Test db mapped?:", test_db in db_paths)
print("Test query result:", run_query(test_sql, test_db))

Databases mapped: 206
Test db mapped?: True
Test query result: {(6,)}


### Build the generation function (question → SQL)

`generate_sql(question, db_id)` formats the prompt IDENTICALLY to training
(same system prompt, same schema+question user turn, same chat template with
`add_generation_prompt=True`), runs greedy decoding (`do_sample=False`) for a
fair, reproducible comparison, and returns just the generated SQL. We also load
the dev split + schema map here (fresh notebook), and test on one example.

In [7]:
from datasets import load_dataset

# Fresh notebook: reload dev questions + schema map
spider_val = load_dataset("xlangai/spider", split="validation")
schema_ds  = load_dataset("richardr1126/spider-schema", split="train")
schema_map = {}
for row in schema_ds:
    txt = f"Tables and columns:\n{row['Schema (values (type))']}"
    if row["Foreign Keys"] and row["Foreign Keys"].strip():
        txt += f"\n\nForeign keys:\n{row['Foreign Keys']}"
    schema_map[row["db_id"]] = txt

SYSTEM_PROMPT = (
    "You are an expert data analyst who writes SQLite SQL queries. "
    "Given a database schema and a natural-language question, respond with a "
    "single valid SQL query that answers it. Use only the tables and columns "
    "in the schema. Output only the SQL query, with no explanation."
)

def generate_sql(question, db_id, max_new_tokens=256):
    user_content = (f"Database schema:\n{schema_map[db_id]}\n\n"
                    f"Question: {question}")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", tokenize=True,
    ).to(model.device)
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tokenizer.eos_token_id)
    # Decode only the newly generated tokens (skip the prompt)
    gen = out[0][inputs.shape[-1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

# Test on one dev example (adapter is currently ON = fine-tuned model)
ex = spider_val[0]
print("Question:", ex["question"])
print("db_id   :", ex["db_id"])
print("Gold SQL:", ex["query"])
print("Model SQL:", generate_sql(ex["question"], ex["db_id"]))

README.md:   0%|          | 0.00/5.51k [00:00<?, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

...

spider_schema_rows_v2.json:   0%|          | 0.00/192k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Question: How many singers do we have?
db_id   : concert_singer
Gold SQL: SELECT count(*) FROM singer


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Model SQL: SELECT count(*) FROM singer


### Evaluate: base vs fine-tuned on 300 dev examples

For each example we generate SQL twice against the same prompt — adapter OFF
(`model.disable_adapter()` = base Qwen) and adapter ON (fine-tuned) — a fully
controlled comparison where only the adapter differs. We score three metrics
for each: execution accuracy (run both queries, compare result sets — the
headline), exact-set match (normalized string equality — stricter), and
valid-SQL % (ran without error — a floor). Runs greedy decoding on 300 examples.

In [8]:
import re, random
from tqdm.auto import tqdm

N = 300
random.seed(42)
indices = random.sample(range(len(spider_val)), N)

def clean_sql(text):
    text = re.sub(r"```sql|```", "", text).strip()  # strip any code fences
    return text.split(";")[0].strip()               # first statement only

def gen(question, db_id, max_new_tokens=256):
    user = f"Database schema:\n{schema_map[db_id]}\n\nQuestion: {question}"
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user}]
    enc = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt",
        return_dict=True,
    ).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tokenizer.eos_token_id)
    gen_ids = out[0][enc["input_ids"].shape[-1]:]
    return clean_sql(tokenizer.decode(gen_ids, skip_special_tokens=True))

def score(pred_sql, gold_sql, db_id):
    pred_res = run_query(pred_sql, db_id)
    gold_res = run_query(gold_sql, db_id)
    exec_ok  = (pred_res is not None) and (pred_res == gold_res)
    valid_ok = (pred_res is not None)
    match_ok = pred_sql.lower().split() == gold_sql.lower().split()
    return exec_ok, valid_ok, match_ok

stats = {"base": [0,0,0], "ft": [0,0,0]}   # [exec, valid, match] counts

for i in tqdm(indices):
    ex = spider_val[i]
    q, db, gold = ex["question"], ex["db_id"], ex["query"]

    with model.disable_adapter():          # adapter OFF = base model
        base_sql = gen(q, db)
    ft_sql = gen(q, db)                     # adapter ON = fine-tuned

    for name, sql in [("base", base_sql), ("ft", ft_sql)]:
        e, v, m = score(sql, gold, db)
        stats[name][0] += e; stats[name][1] += v; stats[name][2] += m

print(f"\n{'Metric':<20}{'Base':>10}{'Fine-tuned':>14}")
for idx, label in [(0,"Execution acc"),(1,"Valid-SQL %"),(2,"Exact match")]:
    b = 100*stats["base"][idx]/N
    f = 100*stats["ft"][idx]/N
    print(f"{label:<20}{b:>9.1f}%{f:>13.1f}%")

  0%|          | 0/300 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2


Metric                    Base    Fine-tuned
Execution acc            64.0%         72.3%
Valid-SQL %              85.3%         91.3%
Exact match              14.7%         37.0%


### Error analysis: are the "wrong" queries actually wrong?

Re-run the fine-tuned model on the same 300 dev examples, but this time record
every prediction. For each one our scorer marked wrong, classify it:
- **invalid**: query errored / hallucinated a column (run_query returned None)
- **valid-but-different**: ran fine but returned different rows than gold

We print counts per bucket + a few examples of each, so we can judge by eye how
many are true model errors vs. artifacts of set-based execution matching
(e.g. column order, extra columns) that penalize correct queries.

In [9]:
import random
random.seed(42)
indices = random.sample(range(len(spider_val)), 300)   # SAME 300 as before

invalid_cases = []          # model SQL didn't run
different_cases = []        # ran, but rows differ from gold
correct = 0

for i in tqdm(indices):
    ex = spider_val[i]
    q, db, gold = ex["question"], ex["db_id"], ex["query"]
    pred = gen(q, db)                       # fine-tuned (adapter ON)

    pred_res = run_query(pred, db)
    gold_res = run_query(gold, db)

    if pred_res is not None and pred_res == gold_res:
        correct += 1
    elif pred_res is None:
        invalid_cases.append((q, gold, pred))
    else:
        different_cases.append((q, gold, pred, gold_res, pred_res))

print(f"Correct (exec match):     {correct}/300 = {100*correct/300:.1f}%")
print(f"Invalid (didn't run):     {len(invalid_cases)}")
print(f"Valid but different rows: {len(different_cases)}\n")

print("="*70)
print("INVALID EXAMPLES (model SQL failed to run):")
for q, gold, pred in invalid_cases[:3]:
    print("-"*70)
    print("Q   :", q)
    print("GOLD:", gold)
    print("PRED:", pred)

print("\n" + "="*70)
print("VALID-BUT-DIFFERENT EXAMPLES (ran, but rows differ):")
for q, gold, pred, gres, pres in different_cases[:5]:
    print("-"*70)
    print("Q        :", q)
    print("GOLD     :", gold)
    print("PRED     :", pred)
    print("GOLD rows:", str(gres)[:120])
    print("PRED rows:", str(pres)[:120])

  0%|          | 0/300 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Correct (exec match):     217/300 = 72.3%
Invalid (didn't run):     26
Valid but different rows: 57

INVALID EXAMPLES (model SQL failed to run):
----------------------------------------------------------------------
Q   : What are the ids and names of all countries that either have more than 3 car makers or produce fiat model ?
GOLD: select t1.countryid ,  t1.countryname from countries as t1 join car_makers as t2 on t1.countryid  =  t2.country group by t1.countryid having count(*)  >  3 union select t1.countryid ,  t1.countryname from countries as t1 join car_makers as t2 on t1.countryid  =  t2.country join model_list as t3 on t2.id  =  t3.maker where t3.model  =  'fiat';
PRED: SELECT T1.CountryName FROM countries AS T1 JOIN car_makers AS T2 ON T1.CountryId  =  T2.CountryId WHERE T2.FullName  =  'Fiat' UNION SELECT T1.CountryName FROM countries AS T1 JOIN car_makers AS T2 ON T1.CountryId  =  T2.CountryId GROUP BY T1.CountryName HAVING COUNT(*)  >  3
------------------------------------

### Quantify metric artifacts: how many failures have degenerate gold?

From the samples, some "wrong" cases are where the GOLD query itself returns
an empty set (e.g. the airport OR-join examples) — the model's query may be
equally or more correct, but set-comparison marks it wrong. We count how many
of the 57 valid-but-different failures fall into this bucket, plus how many are
"model returned empty" (often a wrong value literal like 'Left' vs 'L'). This
tells us roughly how much of the 27.7% gap is true model error vs artifact.

In [10]:
gold_empty = 0        # gold returns nothing -> comparison is degenerate
pred_empty = 0        # model returns nothing (often wrong value literal)
both_nonempty = 0     # both return rows but differ -> most likely real error

for q, gold, pred, gres, pres in different_cases:
    if len(gres) == 0:
        gold_empty += 1
    elif len(pres) == 0:
        pred_empty += 1
    else:
        both_nonempty += 1

total = len(different_cases)
print(f"Valid-but-different breakdown ({total} total):")
print(f"  Gold returns empty (metric artifact):   {gold_empty}")
print(f"  Pred returns empty (often value error): {pred_empty}")
print(f"  Both non-empty, differ (likely real):   {both_nonempty}")

# A rough "adjusted" upper bound: credit the gold-empty cases as not-model's-fault
adjusted = (217 + gold_empty) / 300 * 100
print(f"\nRaw execution accuracy:      72.3%")
print(f"Adjusted upper-bound (crediting {gold_empty} degenerate-gold cases): {adjusted:.1f}%")

Valid-but-different breakdown (57 total):
  Gold returns empty (metric artifact):   3
  Pred returns empty (often value error): 13
  Both non-empty, differ (likely real):   41

Raw execution accuracy:      72.3%
Adjusted upper-bound (crediting 3 degenerate-gold cases): 73.3%
